In [ ]:
# Unified Configuration
# =============================================================================
# All Snowflake names are defined here.
# If anything changes, only edit this file.

SNOWFLAKE_CONFIG = {
    "warehouse": "EAGLE_WH",
    "database":  "CITYLENS_MERGED_DB",
}

# Schema names
SCHEMAS = {
    "housing_serving":    "HOUSING_SERVING",
    "housing_mart":       "HOUSING_MART",
    "housing_core":       "HOUSING_CORE",
    "housing_monitoring": "HOUSING_MONITORING",
    "housing_raw":        "HOUSING_RAW",
    "housing_staging":    "HOUSING_STAGING",

    "transport_serving":  "CITYLENS_SERVING",
    "transport_mart":     "CITYLENS_MART",
    "transport_core":     "CITYLENS_CORE",
    "transport_raw":      "CITYLENS_RAW",
    "transport_staging":  "CITYLENS_STAGING",
    "transport_public":   "CITYLENS_PUBLIC",

    "crime":              "CRIME_PUBLIC",
}

DB = SNOWFLAKE_CONFIG["database"]

# Full table path helpers — use these everywhere instead of hardcoding
class Tables:
    # Housing
    HOUSING_NEIGHBORHOOD_SUMMARY  = f"{DB}.HOUSING_SERVING.SRV_NEIGHBORHOOD_SUMMARY"
    HOUSING_PROPERTY_SUMMARY      = f"{DB}.HOUSING_SERVING.SRV_PROPERTY_PROFILE_SUMMARY"
    HOUSING_QA_CONTEXT            = f"{DB}.HOUSING_SERVING.SRV_HOUSING_QA_CONTEXT"
    HOUSING_MART_EXCEPTIONS       = f"{DB}.HOUSING_MART.MART_TOP_HOUSING_EXCEPTIONS"
    HOUSING_MART_NEIGHBORHOOD     = f"{DB}.HOUSING_MART.MART_NEIGHBORHOOD_PROFILE"
    HOUSING_FACT_PROPERTY         = f"{DB}.HOUSING_CORE.FACT_PROPERTY_VALUE"
    HOUSING_DIM_NEIGHBORHOOD      = f"{DB}.HOUSING_CORE.DIM_NEIGHBORHOOD"
    HOUSING_DIM_PROPERTY_TYPE     = f"{DB}.HOUSING_CORE.DIM_PROPERTY_TYPE"
    HOUSING_QUERY_LOG             = f"{DB}.HOUSING_MONITORING.QUERY_LOG"

    # Transportation
    TRANSPORT_QA_CONTEXT          = f"{DB}.CITYLENS_SERVING.SRV_QA_CONTEXT"
    TRANSPORT_ROUTE_CONTEXT       = f"{DB}.CITYLENS_SERVING.SRV_ROUTE_CONTEXT"
    TRANSPORT_WEATHER_CONTEXT     = f"{DB}.CITYLENS_SERVING.SRV_WEATHER_CONTEXT"
    TRANSPORT_ANOMALY_CONTEXT     = f"{DB}.CITYLENS_SERVING.SRV_ANOMALY_CONTEXT"
    TRANSPORT_STATION_CONTEXT     = f"{DB}.CITYLENS_SERVING.SRV_STATION_CONTEXT"
    TRANSPORT_RELIABILITY         = f"{DB}.CITYLENS_SERVING.SRV_ROUTE_RELIABILITY"
    TRANSPORT_ALERTS_SUMMARY      = f"{DB}.CITYLENS_MART.MART_ALERTS_SUMMARY"
    TRANSPORT_QUERY_LOG           = f"{DB}.CITYLENS_PUBLIC.QUERY_LOG"

    # Crime
    CRIME_CLEAN                   = f"{DB}.CRIME_PUBLIC.CRIME_CLEAN"
    CRIME_SUMMARIES               = f"{DB}.CRIME_PUBLIC.CRIME_SUMMARIES"
    CRIME_MONTHLY                 = f"{DB}.CRIME_PUBLIC.CRIME_MONTHLY"
    POLICY_DOCUMENTS              = f"{DB}.CRIME_PUBLIC.POLICY_DOCUMENTS"

In [ ]:
# CELL 1 — Imports & Session
# ===========================================================================
 
import json
import uuid
import time
from datetime import datetime
from snowflake.snowpark.context import get_active_session
 
session = get_active_session()
session.sql("USE WAREHOUSE EAGLE_WH").collect()
session.sql("USE DATABASE CITYLENS_MERGED_DB").collect()
session.sql("USE SCHEMA HOUSING_SERVING").collect()
 
print(f"✅ Session ready!")
print(f"   Database  : {session.get_current_database()}")
print(f"   Schema    : {session.get_current_schema()}")
print(f"   Warehouse : {session.get_current_warehouse()}")

In [ ]:
# CELL 2 — Config (所有表名在这里，改名字只改这里)
# ===========================================================================
 
DB = "CITYLENS_MERGED_DB"
 
# Housing
HOUSING_NEIGHBORHOOD_SUMMARY = f"{DB}.HOUSING_SERVING.SRV_NEIGHBORHOOD_SUMMARY"
HOUSING_PROPERTY_SUMMARY     = f"{DB}.HOUSING_SERVING.SRV_PROPERTY_PROFILE_SUMMARY"
HOUSING_QA_CONTEXT = f"{DB}.HOUSING_SERVING.SRV_HOUSING_QA_CONTEXT"
HOUSING_MART_EXCEPTIONS      = f"{DB}.HOUSING_MART.MART_TOP_HOUSING_EXCEPTIONS"
HOUSING_MART_NEIGHBORHOOD    = f"{DB}.HOUSING_MART.MART_NEIGHBORHOOD_PROFILE"
HOUSING_FACT_PROPERTY        = f"{DB}.HOUSING_CORE.FACT_PROPERTY_VALUE"
HOUSING_DIM_NEIGHBORHOOD     = f"{DB}.HOUSING_CORE.DIM_NEIGHBORHOOD"
HOUSING_DIM_PROPERTY_TYPE    = f"{DB}.HOUSING_CORE.DIM_PROPERTY_TYPE"
HOUSING_QUERY_LOG            = f"{DB}.HOUSING_MONITORING.QUERY_LOG"
 
print("✅ Config loaded!")
print(f"   DB: {DB}")

In [ ]:
# CELL 3 — Router
# ===========================================================================
 
def route_query(user_query: str) -> dict:
    query_lower = user_query.lower()
 
    # Intent detection
    if any(w in query_lower for w in ['most expensive', 'highest', 'top', 'luxury', 'best']):
        intent = 'ranking_high'
    elif any(w in query_lower for w in ['cheapest', 'lowest', 'affordable', 'cheap', 'least expensive']):
        intent = 'ranking_low'
    elif any(w in query_lower for w in ['why', 'because', 'reason', 'explain']):
        intent = 'diagnostic'
    elif any(w in query_lower for w in ['compare', 'vs', 'versus', 'difference', 'between']):
        intent = 'comparison'
    elif any(w in query_lower for w in ['zip', 'zipcode', 'zip code']):
        intent = 'zipcode'
    elif any(w in query_lower for w in ['condo', 'apartment', 'single family']):
        intent = 'building_type'
    elif any(w in query_lower for w in ['range', 'spread', 'vary', 'old', 'new', 'age']):
        intent = 'value_trend'
    else:
        intent = 'general'
 
    # Entity detection
    neighborhood_names = [
        'allston', 'back bay', 'beacon hill', 'brighton', 'charlestown',
        'chinatown', 'dorchester', 'downtown', 'east boston', 'fenway',
        'harbor islands', 'hyde park', 'jamaica plain', 'longwood',
        'mattapan', 'mission hill', 'north end', 'roslindale', 'roxbury',
        'south boston', 'south end', 'west end', 'west roxbury'
    ]
    if any(n in query_lower for n in neighborhood_names):
        entity_type = 'neighborhood'
    elif any(w in query_lower for w in ['zip', 'zipcode']):
        entity_type = 'zipcode'
    elif any(w in query_lower for w in ['property', 'house', 'home', 'building', 'address', 'condo']):
        entity_type = 'property'
    else:
        entity_type = 'city'
 
    # Analyst selection
    agents = []
    if intent == 'zipcode' or entity_type == 'zipcode':
        agents.append('zipcode_analyst')
    else:
        agents.append('neighborhood_analyst')
 
    if intent in ['ranking_high', 'ranking_low', 'general']:
        agents.append('price_analyst')
 
    if intent == 'building_type' or any(w in query_lower for w in
            ['condo', 'apartment', 'single family', 'commercial', 'residential']):
        agents.append('building_type_analyst')
 
    if intent == 'value_trend' or any(w in query_lower for w in
            ['range', 'vary', 'old', 'new', 'age', 'spread']):
        agents.append('value_trend_analyst')
 
    if entity_type == 'property' or any(w in query_lower for w in
            ['property', 'house', 'home', 'address']):
        agents.append('property_analyst')
 
    routing_plan = {
        'query_id':       str(uuid.uuid4()),
        'user_query':     user_query,
        'intent':         intent,
        'entity_type':    entity_type,
        'time_window':    'FY2025',
        'agents_invoked': agents,
        'query_ts':       datetime.now().isoformat()
    }
 
    print(f"  Intent  : {intent}")
    print(f"  Entity  : {entity_type}")
    print(f"  Agents  : {agents}")
    return routing_plan
 
print("✅ Router defined!")

In [ ]:
# CELL 4 — Analysts
# ===========================================================================
 
def neighborhood_analyst(routing_plan):
    # 用向量相似度检索最相关的 neighborhood
    query_text = routing_plan['user_query'].replace("'", "''")
    results = session.sql(f"""
        SELECT 
            ENTITY_NAME,
            SUMMARY_TEXT,
            VECTOR_COSINE_SIMILARITY(
                EMBEDDING,
                SNOWFLAKE.CORTEX.EMBED_TEXT_768(
                    'snowflake-arctic-embed-m', '{query_text}'
                )
            ) AS SIMILARITY
        FROM {HOUSING_QA_CONTEXT}
        ORDER BY SIMILARITY DESC
        LIMIT 10
    """).to_pandas()
    context_items = [
        {'neighborhood': r['ENTITY_NAME'],
         'summary': r['SUMMARY_TEXT'],
         'similarity': float(r['SIMILARITY'])}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'neighborhood_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
def price_analyst(routing_plan):
    results = session.sql(f"""
        SELECT EXCEPTION_TYPE, NEIGHBORHOOD_NAME, AVG_PROPERTY_VALUE,
               MEDIAN_PROPERTY_VALUE, AVG_PRICE_PER_SQFT, RANK
        FROM {HOUSING_MART_EXCEPTIONS}
        WHERE EXCEPTION_TYPE IN ('TOP_5_MOST_EXPENSIVE', 'BOTTOM_5_LEAST_EXPENSIVE')
        ORDER BY EXCEPTION_TYPE, RANK
    """).to_pandas()
    context_items = [
        {'type': r['EXCEPTION_TYPE'], 'neighborhood': r['NEIGHBORHOOD_NAME'],
         'avg_value': r['AVG_PROPERTY_VALUE'], 'median_value': r['MEDIAN_PROPERTY_VALUE'],
         'price_per_sqft': r['AVG_PRICE_PER_SQFT'], 'rank': r['RANK']}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'price_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def property_analyst(routing_plan):
    results = session.sql(f"""
        SELECT ADDRESS AS ENTITY_NAME, NEIGHBORHOOD_NAME, PROPERTY_TIER,
               VALUE_SCORE, SUMMARY_TEXT
        FROM {HOUSING_PROPERTY_SUMMARY}
        ORDER BY VALUE_SCORE DESC
        LIMIT 10
    """).to_pandas()
    context_items = [
        {'address': r['ENTITY_NAME'], 'neighborhood': r['NEIGHBORHOOD_NAME'],
         'tier': r['PROPERTY_TIER'], 'value_score': r['VALUE_SCORE'], 'summary': r['SUMMARY_TEXT']}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'property_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def zipcode_analyst(routing_plan):
    results = session.sql(f"""
        SELECT ZIPCODE, COUNT(*) AS TOTAL_PROPERTIES,
               ROUND(AVG(TOTAL_VALUE), 2) AS AVG_VALUE,
               ROUND(AVG(PRICE_PER_SQFT), 2) AS AVG_PRICE_PER_SQFT,
               ROUND(MEDIAN(TOTAL_VALUE), 2) AS MEDIAN_VALUE,
               MODE(CITY) AS PRIMARY_AREA
        FROM {HOUSING_FACT_PROPERTY}
        WHERE ZIPCODE IS NOT NULL AND ZIPCODE != '00000'
          AND TOTAL_VALUE > 0 AND PRICE_PER_SQFT IS NOT NULL
        GROUP BY ZIPCODE
        ORDER BY AVG_PRICE_PER_SQFT DESC
        LIMIT 10
    """).to_pandas()
    context_items = [
        {'zipcode': r['ZIPCODE'], 'primary_area': r['PRIMARY_AREA'],
         'total_properties': int(r['TOTAL_PROPERTIES']),
         'avg_value': float(r['AVG_VALUE']),
         'avg_price_per_sqft': float(r['AVG_PRICE_PER_SQFT']),
         'median_value': float(r['MEDIAN_VALUE'])}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'zipcode_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def building_type_analyst(routing_plan):
    results = session.sql(f"""
        SELECT pt.LAND_USE_DESC, pt.CATEGORY,
               COUNT(f.PROPERTY_ID) AS TOTAL_PROPERTIES,
               ROUND(AVG(f.TOTAL_VALUE), 2) AS AVG_VALUE,
               ROUND(AVG(f.PRICE_PER_SQFT), 2) AS AVG_PRICE_PER_SQFT,
               ROUND(AVG(f.LIVING_AREA), 2) AS AVG_LIVING_AREA,
               ROUND(AVG(f.BEDROOMS), 2) AS AVG_BEDROOMS
        FROM {HOUSING_FACT_PROPERTY} f
        JOIN {HOUSING_DIM_PROPERTY_TYPE} pt ON f.PROPERTY_TYPE_KEY = pt.PROPERTY_TYPE_KEY
        WHERE pt.CATEGORY = 'residential'
          AND pt.LAND_USE_DESC IN (
              'RESIDENTIAL CONDO',
              'SINGLE FAM DWELLING',
              'TWO-FAM DWELLING',
              'THREE-FAM DWELLING',
              'APT 4-6 UNITS'
          )
        GROUP BY pt.LAND_USE_DESC, pt.CATEGORY
        ORDER BY AVG_VALUE DESC
    """).to_pandas()
    context_items = [
        {'land_use': r['LAND_USE_DESC'], 'category': r['CATEGORY'],
         'total_properties': int(r['TOTAL_PROPERTIES']),
         'avg_value': float(r['AVG_VALUE']),
         'avg_price_per_sqft': float(r['AVG_PRICE_PER_SQFT']),
         'avg_living_area': float(r['AVG_LIVING_AREA']),
         'avg_bedrooms': float(r['AVG_BEDROOMS'])}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'building_type_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
def value_trend_analyst(routing_plan):
    results = session.sql(f"""
        SELECT n.NEIGHBORHOOD_NAME,
               ROUND(AVG(f.TOTAL_VALUE), 2) AS AVG_VALUE,
               ROUND(MEDIAN(f.TOTAL_VALUE), 2) AS MEDIAN_VALUE,
               ROUND(MAX(f.TOTAL_VALUE), 2) AS MAX_VALUE,
               ROUND(MIN(f.TOTAL_VALUE), 2) AS MIN_VALUE,
               ROUND(STDDEV(f.TOTAL_VALUE), 2) AS STDDEV_VALUE,
               ROUND(AVG(f.PROPERTY_AGE), 2) AS AVG_AGE,
               COUNT(*) AS TOTAL_PROPERTIES
        FROM {HOUSING_FACT_PROPERTY} f
        JOIN {HOUSING_DIM_NEIGHBORHOOD} n ON f.NEIGHBORHOOD_KEY = n.NEIGHBORHOOD_KEY
        GROUP BY n.NEIGHBORHOOD_NAME
        ORDER BY STDDEV_VALUE DESC
        LIMIT 10
    """).to_pandas()
    context_items = [
        {'neighborhood': r['NEIGHBORHOOD_NAME'],
         'avg_value': float(r['AVG_VALUE']), 'median_value': float(r['MEDIAN_VALUE']),
         'max_value': float(r['MAX_VALUE']), 'min_value': float(r['MIN_VALUE']),
         'stddev_value': float(r['STDDEV_VALUE']),
         'avg_age': float(r['AVG_AGE']) if r['AVG_AGE'] else None,
         'total_properties': int(r['TOTAL_PROPERTIES'])}
        for _, r in results.iterrows()
    ]
    return {'analyst': 'value_trend_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
ANALYST_MAP = {
    'neighborhood_analyst':  neighborhood_analyst,
    'price_analyst':         price_analyst,
    'property_analyst':      property_analyst,
    'zipcode_analyst':       zipcode_analyst,
    'building_type_analyst': building_type_analyst,
    'value_trend_analyst':   value_trend_analyst,
}
 
def run_analysts(routing_plan):
    results = {}
    total_retrievals = 0
    for agent_name in routing_plan['agents_invoked']:
        if agent_name in ANALYST_MAP:
            result = ANALYST_MAP[agent_name](routing_plan)
            results[agent_name] = result
            total_retrievals += result['retrieval_count']
            print(f"  ✅ {agent_name} — {result['retrieval_count']} items retrieved")
    return results, total_retrievals
 
print("✅ All analysts defined!")

In [ ]:
# CELL 5 — Answer Generator
# ===========================================================================
 
INTENT_INSTRUCTIONS = {
    'comparison':    "Compare the property types or areas directly. Show specific numbers for each. Highlight key differences.",
    'ranking_high':  "Rank from highest to lowest. List top 3-5 with specific dollar values.",
    'ranking_low':   "Rank from lowest to highest. List bottom 3-5 with specific dollar values.",
    'zipcode':       "Focus on zipcode-level data. Give exact zipcode numbers and their values. Rank them clearly.",
    'diagnostic':    "Explain the reasons using the data. Be specific about which factors drive the outcome.",
    'building_type': "Compare building types directly with specific numbers for each type.",
    'value_trend':   "Describe the spread and variation. Include min, max, and standard deviation where available.",
    'general':       "Answer directly with specific data points. Include neighborhood names and dollar amounts.",
}
 
def generate_answer(routing_plan, analyst_results):
    context_text = ""
    for analyst_name, result in analyst_results.items():
        context_text += f"\n=== {analyst_name.upper()} ===\n"
        for item in result['context']:
            context_text += json.dumps(item) + "\n"
    context_text = context_text[:5000]
 
    intent = routing_plan.get('intent', 'general')
    task_instruction = INTENT_INSTRUCTIONS.get(intent, INTENT_INSTRUCTIONS['general'])
 
    prompt = f"""You are a Boston Housing Intelligence Assistant.
 
Task: {task_instruction}
 
Question: {routing_plan['user_query']}
 
Data (use ONLY the data provided below, do not make up neighborhoods or values):
{context_text}

IMPORTANT: Only use neighborhood names and values that appear in the data above. Do not invent data.
 
Structure your answer as:
1. DIRECT ANSWER (1-2 sentences with specific numbers)
2. KEY FINDINGS (3-5 bullet points with data)
3. INSIGHT (1 sentence conclusion)"""
 
    start_time = time.time()
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5' , $${prompt}$$) AS ANSWER
    """).collect()
    latency_ms = int((time.time() - start_time) * 1000)
 
    return {
        'answer':     result[0]['ANSWER'],
        'tokens_in':  0,
        'tokens_out': 0,
        'latency_ms': latency_ms
    }
 
print("✅ Answer generator defined!")

In [ ]:
# CELL 6 — Reflection & Logger
# ===========================================================================
 
BOSTON_NEIGHBORHOODS = [
    'south boston', 'downtown', 'fenway', 'beacon hill', 'back bay',
    'chinatown', 'west end', 'north end', 'dorchester', 'roxbury',
    'hyde park', 'mattapan', 'allston', 'brighton', 'charlestown',
    'east boston', 'jamaica plain', 'roslindale', 'west roxbury'
]
 
def reflect_on_answer(answer, user_query):
    score = 0
    if any(char.isdigit() for char in answer):
        score += 30
    if any(n in answer.lower() for n in BOSTON_NEIGHBORHOODS):
        score += 40
    if 100 < len(answer) < 800:
        score += 30
    return score
 
def log_query(routing_plan, answer_result, reflection_score, total_retrievals):
    try:
        # 清理答案里的特殊字符，防止 SQL 报错
        clean_answer = answer_result['answer'][:500]
        clean_answer = clean_answer.replace("'", "''").replace("$", "\\$")
        
        session.sql(f"""
            INSERT INTO {HOUSING_QUERY_LOG} (
                QUERY_ID, USER_QUERY, QUERY_TS, INTENT, ENTITY_TYPE,
                ENTITY_NAME, TIME_WINDOW, ROUTE_TAKEN, AGENTS_INVOKED,
                RETRIEVAL_COUNT, TOKENS_IN, TOKENS_OUT, TOTAL_LATENCY_MS,
                FINAL_ANSWER, FINAL_ANSWER_SCORE, GROUNDEDNESS_SCORE
            ) VALUES (
                '{routing_plan['query_id']}',
                '{routing_plan['user_query'].replace("'", "''")}',
                CURRENT_TIMESTAMP(),
                '{routing_plan['intent']}',
                '{routing_plan['entity_type']}',
                'Boston',
                '{routing_plan['time_window']}',
                'multi_analyst',
                '{",".join(routing_plan['agents_invoked'])}',
                {total_retrievals},
                {answer_result['tokens_in']},
                {answer_result['tokens_out']},
                {answer_result['latency_ms']},
                '{clean_answer}',
                {reflection_score},
                {reflection_score}
            )
        """).collect()
        print(f"  ✅ Query logged: {routing_plan['query_id']}")
    except Exception as e:
        print(f"  ⚠️  Log failed: {e}")
 
print("✅ Reflection & Logger defined!")

In [ ]:
# CELL 7 — Main Agent Function
# ===========================================================================
 
def run_housing_agent(user_query: str) -> str:
    print(f"\n{'='*60}")
    print(f"❓ {user_query}")
    print('='*60)
 
    print("\n[Step 1] Routing...")
    routing_plan = route_query(user_query)
 
    print("\n[Step 2] Running analysts...")
    analyst_results, total_retrievals = run_analysts(routing_plan)
 
    print("\n[Step 3] Generating answer...")
    answer_result = generate_answer(routing_plan, analyst_results)
 
    print("\n[Step 4] Reflecting...")
    reflection_score = reflect_on_answer(answer_result['answer'], user_query)
    print(f"  Score: {reflection_score}/100")
 
    print("\n[Step 5] Logging...")
    log_query(routing_plan, answer_result, reflection_score, total_retrievals)
 
    print(f"\n{'='*60}")
    print("🤖 FINAL ANSWER:")
    print(answer_result['answer'])
    print(f"\n⏱  Latency    : {answer_result['latency_ms']}ms")
    print(f"📊 Retrievals : {total_retrievals}")
    print(f"⭐ Score      : {reflection_score}/100")
 
    return answer_result['answer']
 
print("✅ Housing agent ready!")

In [ ]:
# CELL 8 — Test
# ===========================================================================
 
run_housing_agent("What are the most expensive neighborhoods in Boston?")

In [ ]:
test_questions = [
    "What are the most expensive neighborhoods in Boston?",
    "Which zipcode has the highest price per sqft?",
    "How do condos compare to single family homes in value?",
    "Which neighborhoods have the widest range of property values?",
    "What are the most affordable areas in Boston?",
]

for q in test_questions:
    run_housing_agent(q)
    print()

In [ ]:
run_housing_agent("How do condos compare to single family homes in value?")